<a href="https://colab.research.google.com/github/ryancookd/RNA-3D-Folding/blob/initial-commit/RNA_3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

stanford_rna_3d_folding_2_path = kagglehub.competition_download('stanford-rna-3d-folding-2')

print('Data source import complete.')


In [ ]:
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf

In [ ]:
input_root = "/kaggle/input"
folders = [os.path.join(input_root, d) for d in os.listdir(input_root)]

DATA_DIR = None
for f in folders:
    if os.path.isdir(f) and os.path.exists(os.path.join(f, "sample_submission.csv")):
        DATA_DIR = f
        break

print("Detected DATA_DIR:", DATA_DIR)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find a folder in /kaggle/input that contains sample_submission.csv")

print("Top-level files:", sorted(os.listdir(DATA_DIR))[:30])

SEED = 42
rng = np.random.default_rng(SEED)

In [ ]:
MAX_LEN = 1024        # try 1536 if your session has enough RAM/GPU
N_TARGETS = 2000      # increase as memory allows
EPOCHS = 10
BATCH = 4             # Kabsch + dropout => smaller batch
print("MAX_LEN", MAX_LEN, "N_TARGETS", N_TARGETS, "EPOCHS", EPOCHS, "BATCH", BATCH)

In [ ]:
train_seq = pd.read_csv(os.path.join(DATA_DIR, "train_sequences.csv"))
train_seq["seq_len"] = train_seq["sequence"].str.len()

usecols = ["ID", "resid", "x_1", "y_1", "z_1"]
train_lbl = pd.read_csv(
    os.path.join(DATA_DIR, "train_labels.csv"),
    usecols=usecols,
    low_memory=False
)
train_lbl["target_id"] = train_lbl["ID"].str.split("_").str[0]

print("train_sequences:", train_seq.shape)
print("train_labels   :", train_lbl.shape)

In [ ]:
good_targets = train_seq.loc[train_seq["seq_len"] <= MAX_LEN, "target_id"].drop_duplicates()
good_targets = good_targets.iloc[:N_TARGETS]
print("Using targets:", len(good_targets))

df = train_lbl[train_lbl["target_id"].isin(good_targets)].merge(
    train_seq[["target_id", "sequence", "seq_len"]],
    on="target_id",
    how="left"
)
df = df.dropna(subset=["sequence"]).sort_values(["target_id", "resid"]).reset_index(drop=True)
print("Training rows:", len(df), "targets:", df["target_id"].nunique())

In [ ]:
base_to_id = {"A": 0, "C": 1, "G": 2, "U": 3}
UNK = 4

def encode(seq: str) -> np.ndarray:
    return np.array([base_to_id.get(ch, UNK) for ch in seq], dtype=np.int32)

tokens_list, coords_list = [], []
for tid, g in df.groupby("target_id", sort=False):
    seq = g["sequence"].iloc[0]
    tok = encode(seq)               # guaranteed len <= MAX_LEN
    coords = g[["x_1", "y_1", "z_1"]].to_numpy(np.float32)

    if len(tok) != len(coords):
        continue

    tokens_list.append(tok)
    coords_list.append(coords)

print("Examples:", len(tokens_list))

X = np.full((len(tokens_list), MAX_LEN), UNK, dtype=np.int32)
Y = np.zeros((len(coords_list), MAX_LEN, 3), dtype=np.float32)
M = np.zeros((len(coords_list), MAX_LEN), dtype=np.float32)

for i, (tok, coords) in enumerate(zip(tokens_list, coords_list)):
    L = len(tok)
    X[i, :L] = tok
    Y[i, :L, :] = coords
    M[i, :L] = 1.0

# Clean NaNs by masking out non-finite coordinate rows
finite = np.isfinite(Y).all(axis=-1)
M = M * finite.astype(np.float32)
Y[~finite] = 0.0

print("X:", X.shape, "Y:", Y.shape, "M:", M.shape)
print("Active points:", float(M.sum()))


In [ ]:
ds = tf.data.Dataset.from_tensor_slices((X, Y, M))
ds = ds.shuffle(4096, seed=SEED).batch(BATCH).prefetch(tf.data.AUTOTUNE)

VOCAB_SIZE = 5
EMB = 96

inp = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32)
x = tf.keras.layers.Embedding(VOCAB_SIZE, EMB)(inp)
x = tf.keras.layers.Dropout(0.2)(x)

x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(192, return_sequences=True))(x)
x = tf.keras.layers.Dropout(0.2)(x)

x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)

pred = tf.keras.layers.Dense(3)(x)
model = tf.keras.Model(inp, pred)

def kabsch_align(P, Q, mask):
    # P,Q: (B,L,3) ; mask: (B,L)
    mask3 = tf.expand_dims(mask, -1)
    wsum = tf.reduce_sum(mask3, axis=1, keepdims=True) + 1e-6

    Pc = tf.reduce_sum(P * mask3, axis=1, keepdims=True) / wsum
    Qc = tf.reduce_sum(Q * mask3, axis=1, keepdims=True) / wsum

    P0 = (P - Pc) * mask3
    Q0 = (Q - Qc) * mask3

    H = tf.matmul(P0, Q0, transpose_a=True)  # (B,3,3)
    S, U, Vt = tf.linalg.svd(H, full_matrices=False)
    V = tf.transpose(Vt, perm=[0, 2, 1])

    d = tf.linalg.det(tf.matmul(V, U, transpose_b=True))
    d = tf.reshape(d, (-1, 1, 1))

    eye = tf.eye(3, batch_shape=[tf.shape(P)[0]])
    corr = tf.concat([eye[:, :, :2], eye[:, :, 2:3] * tf.sign(d)], axis=2)

    R = tf.matmul(tf.matmul(V, corr), U, transpose_b=True)
    return tf.matmul(P - Pc, R) + Qc

def masked_aligned_mse(y_true, y_pred, mask):
    y_pred_aligned = kabsch_align(y_pred, y_true, mask)
    mask3 = tf.expand_dims(mask, -1)
    se = tf.square(y_true - y_pred_aligned) * mask3
    return tf.reduce_sum(se) / (tf.reduce_sum(mask3) + 1e-6)

opt = tf.keras.optimizers.Adam(1e-3, clipnorm=1.0)

@tf.function
def train_step(xb, yb, mb):
    with tf.GradientTape() as tape:
        yp = model(xb, training=True)
        loss = masked_aligned_mse(yb, yp, mb)
    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    return loss

for epoch in range(EPOCHS):
    losses = []
    for xb, yb, mb in ds:
        losses.append(train_step(xb, yb, mb).numpy())
    print(f"epoch {epoch+1}/{EPOCHS}: aligned_loss={float(np.mean(losses)):.4f}")


In [ ]:
test_seq = pd.read_csv(os.path.join(DATA_DIR, "test_sequences.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
sub = sample_sub.copy()

coord_cols = [c for c in sub.columns if c.startswith(("x_", "y_", "z_"))]
sub[coord_cols] = sub[coord_cols].astype(np.float32)

print("Test targets:", test_seq["target_id"].nunique(), "rows:", len(test_seq))
print("Submission rows:", len(sub), "coord cols:", len(coord_cols))

pred5 = {}  # target_id -> (5, L, 3)

for _, row in test_seq.iterrows():
    tid = row["target_id"]
    seq = row["sequence"]
    tok = encode(seq)
    L_full = len(tok)

    # If test sequence is longer than MAX_LEN, truncate (keeps code safe)
    tok_use = tok[:MAX_LEN]
    L = len(tok_use)

    x_in = np.full((1, MAX_LEN), UNK, dtype=np.int32)
    x_in[0, :L] = tok_use

    samples = []
    for s in range(5):
        yhat = model(x_in, training=True).numpy()[0]  # dropout active
        samples.append(yhat[:L].astype(np.float32))

    # Store full length (pad remaining residues with zeros if truncated)
    arr = np.zeros((5, L_full, 3), dtype=np.float32)
    arr[:, :L, :] = np.stack(samples, axis=0)
    pred5[tid] = arr

print("Predicted targets:", len(pred5))

# Parse submission IDs -> target_id and resid
tid_res = sub["ID"].str.rsplit("_", n=1, expand=True)
sub["_target_id"] = tid_res[0]
sub["_resid"] = tid_res[1].astype(int)

# Initialize coords
for k in [1,2,3,4,5]:
    sub[f"x_{k}"] = 0.0
    sub[f"y_{k}"] = 0.0
    sub[f"z_{k}"] = 0.0

# Fill per target
for tid, g in sub.groupby("_target_id", sort=False):
    if tid not in pred5:
        continue
    arr = pred5[tid]  # (5,L,3)
    idx = g["_resid"].to_numpy() - 1
    L = arr.shape[1]
    ok = (idx >= 0) & (idx < L)

    for k in range(5):
        xyz = np.zeros((len(idx), 3), dtype=np.float32)
        xyz[ok] = arr[k, idx[ok], :]
        sub.loc[g.index, [f"x_{k+1}", f"y_{k+1}", f"z_{k+1}"]] = xyz

sub.drop(columns=["_target_id","_resid"], inplace=True)

zeros = (sub[["x_1","y_1","z_1"]].to_numpy() == 0).all(axis=1).sum()
print("Rows with x_1,y_1,z_1 all zero:", int(zeros), "out of", len(sub))

sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv")
sub.head()